In [ ]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from itertools import product
from datetime import datetime
import time

# Network

In [ ]:
# Load Network Files
network_name = "11-500"
network_path = f"data/network/{network_name}/"
node_df = pd.read_csv(network_path + "nodes.csv")
link_df = pd.read_csv(network_path + "edges.csv")

node_id_list = node_df['node_index'].tolist()
print(f"Number of nodes: {len(node_id_list)}")

left_node_id_list = [i for i in range(0, len(node_id_list)//2)]
right_node_id_list = [i for i in range(len(node_id_list)//2, len(node_id_list))]
print(f"Number of left nodes: {len(left_node_id_list)}")
print(f"Number of right nodes: {len(right_node_id_list)}")

left_mobility_node_id_list = [116, 117, 118, 119, 120]
right_mobility_node_id_list = [237, 238, 239, 240, 241]

In [ ]:
# Load Distance Matrix
distance_matrix = np.load(network_path + "dist_matrix.npy")
print(f"Distance matrix shape: {distance_matrix.shape}")

In [ ]:
# Walking Speed (m/s)
WALKING_SPEED = 1.33

# Street station transfer time (s)
STREET_STATION_TRANSFER_TIME = 60

# Create Walking Time Matrix (s)
walking_time_matrix = distance_matrix / WALKING_SPEED

# PT Router

In [ ]:
from src.pt.PTOperator import PTOperator

train_10_gtfs_dir = r"data/gtfs/train/train_headway_10/matched"
train_10_pt_control = PTOperator(train_10_gtfs_dir)

train_20_gtfs_dir = r"data/gtfs/train/train_headway_20/matched"
train_20_pt_control = PTOperator(train_20_gtfs_dir)

train_30_gtfs_dir = r"data/gtfs/train/train_headway_30/matched"
train_30_pt_control = PTOperator(train_30_gtfs_dir)

# Variables

In [ ]:
# Train Headway (min)
train_headway_list = [10, 20, 30]

# MoD Fleet Size
mod_fleet_size_list = [30, 50, 70, 90, 110, 130, 150]

# MaaS Platform Communication Strategy
maas_communication_strategy_list = ['default', 'TPCS']

# Random Seed
random_seed_list = [3, 6, 9]

# MoD Waiting Time Threshold (s)
mod_waiting_time_threshold_list = [300, 600, 900]
# MoD Detour Time Threshold (%)
mod_detour_time_threshold_list = [30, 60, 90]

# Demand Size
demand_size_list = [i for i in range(100, 1001, 100)]
# Demand Split Ratio (Intra Modal, %)
demand_split_ratio_list = [0, 20, 40, 60, 80]

# Total Simulation Time (s)
total_sim_time = [0, 10800]  # 3 hours
# Warm-up Time (s)
warmup_time = 3600  # 1 hour
# Simulation Time Period (s)
time_period = [warmup_time, total_sim_time[1]+warmup_time]  # 1h + 3h

amod_request_level_analysis_folder = "data/amod-sim-results"

demand_files_folder = "data/demand/11-500/amod"

In [ ]:
# All scenario combinations
all_scenario_combinations = list(product(
    random_seed_list,
    mod_fleet_size_list,
    demand_size_list,
    demand_split_ratio_list,
    maas_communication_strategy_list,
    mod_detour_time_threshold_list,
    mod_waiting_time_threshold_list,
    train_headway_list
))

In [ ]:
for scenario_combination in tqdm(all_scenario_combinations):
    (
        random_seed,
        fleet_size,
        demand_size,
        demand_split_ratio,
        broker_type,
        op_max_detour_time_factor,
        op_max_wait_time,
        train_headway
    ) = scenario_combination

    demand_filepath = os.path.join(demand_files_folder, f"amod_ds{demand_size}_dsr{demand_split_ratio}_rs{random_seed}.csv")

    scenario_name = f"amod-{demand_size}-{demand_split_ratio}-{fleet_size}-{broker_type}-{op_max_detour_time_factor}-{op_max_wait_time}-{train_headway}-{random_seed}-{time_period[0]}-{time_period[1]}"

    amod_request_level_analysis_results_filepath = os.path.join(amod_request_level_analysis_folder, scenario_name, 'amod_request_level_analysis_results.csv')

    # Load files
    demand = pd.read_csv(demand_filepath)
    amod_request_level_analysis_results = pd.read_csv(amod_request_level_analysis_results_filepath)

    # Add new column
    amod_request_level_analysis_results['served_by_walking'] = 0

    # Select all unserved requests
    unserved_requests = amod_request_level_analysis_results[amod_request_level_analysis_results['served_by_amod'] == False]

    # Get unserved intra requests
    unserved_intra_requests = unserved_requests[unserved_requests['rq_type'] == 'intra']
    # Get unserved inter requests
    unserved_inter_requests = unserved_requests[unserved_requests['rq_type'] == 'inter']

    # Process unserved intra requests
    for idx, unserved_request in unserved_intra_requests.iterrows():
        request_id = unserved_request['request_id']
        origin_node = demand.loc[demand['request_id'] == request_id, 'start'].values[0]
        destination_node = demand.loc[demand['request_id'] == request_id, 'end'].values[0]
        walking_time = walking_time_matrix[origin_node, destination_node]
        # Update the result dataframe
        amod_request_level_analysis_results.loc[amod_request_level_analysis_results['request_id'] == request_id, 'served_by_walking'] = 1
        amod_request_level_analysis_results.loc[amod_request_level_analysis_results['request_id'] == request_id, 'total_journey_time'] = walking_time
    
    # Process unserved inter requests
    for idx, unserved_request in unserved_inter_requests.iterrows():
        request_id = unserved_request['request_id']
        origin_node = demand.loc[demand['request_id'] == request_id, 'start'].values[0]
        destination_node = demand.loc[demand['request_id'] == request_id, 'end'].values[0]
        subnetwork = unserved_request['subnetwork']
        request_time = demand.loc[demand['request_id'] == request_id, 'rq_time'].values[0]

        if subnetwork == 'left':
            start_station_street_node = 120
            start_station_id = "MH-L"
            end_station_street_node = 241
            end_station_id = "MH-R"
        else:
            start_station_street_node = 241
            start_station_id = "MH-R"
            end_station_street_node = 120
            end_station_id = "MH-L"
        walking_time_to_station = walking_time_matrix[origin_node, start_station_street_node] + STREET_STATION_TRANSFER_TIME
        walking_time_from_station = walking_time_matrix[end_station_street_node, destination_node] + STREET_STATION_TRANSFER_TIME

        arrival_time_at_station = request_time + walking_time_to_station

        # Conver to arrival datetime
        arrival_datetime = datetime(2024, 1, 1, 0, 0, 0) + pd.to_timedelta(arrival_time_at_station, unit='s')

        if train_headway == 10:
            pt_control = train_10_pt_control
        elif train_headway == 20:
            pt_control = train_20_pt_control
        else:
            pt_control = train_30_pt_control

        duration = pt_control.return_fastest_pt_journey_1to1(start_station_id, end_station_id, arrival_datetime, 3, detailed=False)['duration']

        # Calculate total journey time
        total_journey_time = walking_time_to_station + duration + walking_time_from_station
        # Update the result dataframe
        amod_request_level_analysis_results.loc[amod_request_level_analysis_results['request_id'] == request_id, 'served_by_walking'] = 1
        amod_request_level_analysis_results.loc[amod_request_level_analysis_results['request_id'] == request_id, 'total_journey_time'] = total_journey_time

    # Save the updated results
    amod_request_level_analysis_results.to_csv(amod_request_level_analysis_results_filepath, index=False)

